# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [1]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [3]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [4]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'467960c3-e9a0-4279-a7d9-fdd0084eb97a': {'version…

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Exercises General Overview

1. Load environment variables and initialize OpenAI client
    - Read API key from .env
    - Create OpenAI client instance

2. Define a function to send the full conversation history to GPT
    - Input: list of messages (context)
    - Output: assistant response
    - Use ChatCompletion with model gpt-3.5-turbo

3. Define a function to handle user interaction
    - Read user input from Panel widget
    - Append user message to context
    - Call GPT using the conversation history
    - Append assistant response to context
    - Update the UI with both messages

4. Create the system prompt (3 different versions)
    - Version 1: Classic structured OrderBot
    - Version 2: Sales-oriented OrderBot
    - Version 3: Minimalist concise OrderBot

5. Build the Panel interface
    - Text input for user messages
    - Button to trigger conversation
    - Column to display chat history

6. Bind the button to the conversation function
    - Each click sends the message and updates the chat


## Exercise 1 — Classic OrderBot Prompt

Initialize OpenAI client and load API key.
Create a structured system prompt with clear steps for the OrderBot.
Set up Panel UI: text input, button, and chat display column.

When user sends a message:
    - Add message to conversation context
    - Send full context to GPT
    - Receive assistant reply
    - Append reply to context and update UI
    
Run the dashboard to interact with the classic, reliable OrderBot.


In [12]:
# orderbot_v1_classic.py

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
import panel as pn  # GUI

# Load environment variables
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# OpenAI client
client = OpenAI(
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

def add_prompts_conversation(_):
    # Get user input
    prompt = client_prompt.value_input
    client_prompt.value = ''

    # Add user message to context
    context.append({'role': 'user', 'content': f"{prompt}"})

    # Get model response
    response = continue_conversation(context)

    # Add assistant response to context
    context.append({'role': 'assistant', 'content': f"{response}"})

    # Update UI panels
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600))
    )

    return pn.Column(*panels)

# ---------- SYSTEM PROMPT V1: CLASSIC ORDERBOT ----------

context = [{
    'role': 'system',
    'content': """
Act as OrderBot for a delivery-only restaurant called My Dear Frankfurt.
Your job is to:
1) Greet the customer warmly.
2) Collect the full order, including drinks.
3) Ask clarifying questions about sizes, toppings, and extras.
4) Summarize the complete order.
5) Ask for confirmation.
6) Collect payment.

Keep responses short, friendly, and efficient.
Never invent menu items. Use ONLY this menu:

Burgers: 12.95, 10.00, 7.00
Frankfurts: 10.95, 9.25, 6.50
Sandwiches: 11.95, 9.75, 6.75
Fries: 4.50, 3.50
Salad: 7.25

Toppings:
extra cheese 2.00
mushrooms 1.50
martra sausage 3.00
canadian bacon 3.50
romesco sauce 1.50
peppers 1.00

Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
vichy catalan 5.00
"""
}]

# ---------- PANEL UI ----------

pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'b1c3b7c6-93d6-486c-bd1a-b349a1d91b96': {'version…

## Exercise 2 Sales‑Oriented OrderBot Prompt

Initialize OpenAI client and load API key.
Create a persuasive system prompt encouraging upselling and suggestions.
Build Panel UI for user input and chat history.

On each user message:
    - Add message to context
    - Call GPT with slightly higher temperature for creativity
    - Append assistant response to context
    - Update chat panel with both messages
    
Run the dashboard to observe how the bot behaves when selling proactively.



In [14]:
# orderbot_v2_sales.py

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
import panel as pn  # GUI

# Load environment variables
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# OpenAI client
client = OpenAI(
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0.3):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

def add_prompts_conversation(_):
    prompt = client_prompt.value_input
    client_prompt.value = ''

    context.append({'role': 'user', 'content': f"{prompt}"})

    response = continue_conversation(context)

    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600))
    )

    return pn.Column(*panels)

# ---------- SYSTEM PROMPT V2: SALES-ORIENTED ORDERBOT ----------

context = [{
    'role': 'system',
    'content': """
You are OrderBot, a friendly and persuasive sales assistant for My Dear Frankfurt.
Your goals:
- Greet the customer with high energy.
- Suggest popular combinations (but ONLY using items from the menu).
- Offer upsells such as toppings or drinks.
- Ask questions to avoid ambiguity (sizes, toppings, combos).
- Summarize the order clearly.
- Confirm before payment.

IMPORTANT:
Do NOT invent items.
Do NOT change prices.
If the customer asks for something not on the menu, politely decline and offer alternatives.

Menu:

Burgers: 12.95, 10.00, 7.00
Frankfurts: 10.95, 9.25, 6.50
Sandwiches: 11.95, 9.75, 6.75
Fries: 4.50, 3.50
Salad: 7.25

Toppings:
extra cheese 2.00
mushrooms 1.50
martra sausage 3.00
canadian bacon 3.50
romesco sauce 1.50
peppers 1.00

Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
vichy catalan 5.00
"""
}]

# ---------- PANEL UI ----------

pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'dbe52632-22a5-480a-b1d0-1568de192848': {'version…

## Exercise 3 Minimalist OrderBot Prompt

Initialize OpenAI client and load API key.
Create a very concise system prompt with strict step-by-step behavior.
Build Panel UI for user input and chat display.

On each user message:
    - Add message to context
    - Call GPT with low temperature for strict, controlled output
    - Append assistant response
    - Update chat panel
    
Run the dashboard to test how the bot performs with minimal instructions.


In [11]:
# orderbot_v3_minimal.py

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
import panel as pn  # GUI

# Load environment variables
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# OpenAI client
client = OpenAI(
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

def add_prompts_conversation(_):
    prompt = client_prompt.value_input
    client_prompt.value = ''

    context.append({'role': 'user', 'content': f"{prompt}"})

    response = continue_conversation(context)

    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600))
    )

    return pn.Column(*panels)

# ---------- SYSTEM PROMPT V3: MINIMALIST ORDERBOT ----------

context = [{
    'role': 'system',
    'content': """
You are OrderBot for My Dear Frankfurt.
Be extremely concise.
Follow this sequence only:
1) Greet.
2) Ask for the order.
3) Ask clarifying questions.
4) Summarize.
5) Confirm.
6) Charge.

Use ONLY the menu below. If the user asks for something else, say it is unavailable.

Menu:

Burgers: 12.95, 10.00, 7.00
Frankfurts: 10.95, 9.25, 6.50
Sandwiches: 11.95, 9.75, 6.75
Fries: 4.50, 3.50
Salad: 7.25

Toppings:
extra cheese 2.00
mushrooms 1.50
martra sausage 3.00
canadian bacon 3.50
romesco sauce 1.50
peppers 1.00

Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
vichy catalan 5.00
"""
}]

# ---------- PANEL UI ----------

pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'35fc5cf9-a4d1-4e74-a393-c2aab1946d97': {'version…

## Conclusion — Comparison of the Three Prompt Versions

The three prompt versions produced noticeably different behaviors, demonstrating how prompt design directly shapes the performance and reliability of a GPT‑based agent.

1. Classic OrderBot Prompt
This version was the most stable, predictable, and accurate.
Because the instructions were detailed and structured, the model followed the expected workflow consistently: greeting → order collection → clarifications → summary → confirmation → payment.
It avoided inventing menu items and maintained a friendly but controlled tone.
This prompt is the best option when reliability and consistency are required.

2. Sales‑Oriented OrderBot Prompt
This version generated a more energetic, persuasive, and proactive assistant.
It successfully suggested extras and drinks, but the increased creativity also led to occasional issues, such as recommending combinations that were not explicitly listed on the menu.
This shows that when creativity increases, the risk of hallucinations also increases.
It is useful for commercial scenarios but requires strict constraints to avoid errors.

3. Minimalist OrderBot Prompt
The minimalist prompt produced a bot that was fast and concise, but also the least reliable.
Because the instructions were short, the model sometimes skipped steps, asked fewer clarifying questions, or summarized too early.
The lack of structure reduced its ability to maintain a complete and orderly conversation flow.
This version works for simple interactions but sacrifices precision and robustness.

Final Summary
The experiment clearly shows that prompt detail directly affects model behavior:

More structure → more control, fewer mistakes

More creativity → more risk of hallucinations

Fewer instructions → more inconsistency and missing steps

Overall, the exercise demonstrates that effective prompt engineering is not just about giving instructions, but about designing a clear behavioral framework that guides the model toward reliable and predictable outputs.